# 패키지 및 데이터

In [29]:
# ==================================================
# 기본 라이브러리
# ==================================================
import numpy as np
import seaborn as sb
import pandas as pd

from matplotlib import pyplot as plt
from pandas import DataFrame, concat



# ==================================================
# scikit-learn 공통
# ==================================================
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    learning_curve,
)

from sklearn.preprocessing import StandardScaler


# ==================================================
# 분류 모델
# ==================================================
from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier,
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    log_loss,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)


# ==================================================
# Boosting 계열
# ==================================================
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# ==================================================
# 사용자 정의 모듈
# ==================================================
from hossam import *


### 데이터

In [30]:
origin = pd.read_csv("kta-county-data-december-2025.csv")
origin.head()
origin.info()
origin.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94923 entries, 0 to 94922
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   RPT_DATE      94923 non-null  object 
 1   RPT_YR        94923 non-null  object 
 2   GEOGRAPHY     94923 non-null  object 
 3   SVC_DESC      94923 non-null  object 
 4   AMOUNT        62812 non-null  object 
 5   AMOUNT_ANNOT  32119 non-null  float64
 6   AMOUNT_TYPE   94903 non-null  object 
dtypes: float64(1), object(6)
memory usage: 5.1+ MB


(94923, 7)

In [ ]:
df = origin.copy()
df.describe()

,AMOUNT_ANNOT
count,32119.000
mean,1.076
std,0.265
min,1.000
25%,1.000
50%,1.000
75%,1.000
max,2.000


In [32]:
df.isna().sum()
df.dropna()



,RPT_DATE,RPT_YR,GEOGRAPHY,SVC_DESC,AMOUNT,AMOUNT_ANNOT,AMOUNT_TYPE
68668,6/6/2024,FY2223,Butte,CRISIS_INTERVENTION,,2.000,MEMBERS
68900,6/6/2024,FY2223,Kern,CRISIS_STABILIZATION,,2.000,MEMBERS
68901,6/6/2024,FY2223,Kern,CRISIS_STABILIZATION,,2.000,HOURS
68902,6/6/2024,FY2223,Kern,CRISIS_STABILIZATION,,2.000,DOLLARS
69009,6/6/2024,FY2223,Madera,ICC,,2.000,MEMBERS
69402,6/6/2024,FY2223,San Diego,SDMC_HOSPITAL_INPATIENT,,2.000,MEMBERS
69403,6/6/2024,FY2223,San Diego,SDMC_HOSPITAL_INPATIENT,,2.000,DAYS
69404,6/6/2024,FY2223,San Diego,SDMC_HOSPITAL_INPATIENT,,2.000,DOLLARS


# 타입변환

결측치를 무조건 삭제하면 데이터의 30% 이상이 날아가 분석결과가 왜도될 수 있기 때문에, 특정 칼럼을 숫자형으로 변환한다.

In [33]:
# 1. pd.to_numeric함수에서 AMOUNT 컬럼의 콤마(,) 제거 및 숫자형 변환
df['AMOUNT'] = df['AMOUNT'].str.replace(',', '')
df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')

# 2. 결측치 확인
print(df["AMOUNT"].isnull().sum())

32119


### AMOUNT_ANNOT를 0이나 중앙값으로 대체할지 결정<br>
- 불확실한 값을 임의로 채우면 변수 간의 관계가 왜곡될 수 있다
- 따라서 결측치 삭제를 할 것이다


### [1] AMOUNT칼럼의 NaN값을 모두 0으로 채운다

In [42]:
df["AMOUNT"]= pd.to_numeric(df["AMOUNT"], errors='coerce').fillna(0)

### [2] AMOUNT_ANNOT칼럼의 NaN값을 모두 0으로 채운다

In [45]:
df["AMOUNT_ANNOT"] = df["AMOUNT_ANNOT"].fillna(0)

# 실제 분석을 위한 수칳여 칼럼 생성(AMOUNT)열 처리
df["AMOUNT_NUM"] = pd.to_numeric(df["AMOUNT"], errors='coerce')



### [3] AMOUNT_TYPE의 NaN값 삭제
> 삭제하는 이유: 예를 들어 AMOUNT값이 존재하더라도 AMOUNT_TYPE에서 빈 값이 발견된다면, 해당 값이 무엇을 의미하는지 모르기 때문에 분석 결과의 신뢰도가 떨어진다.

> 따라서, 결측치각 20여 건으로 전체 데이터 대비 매우 적어, 결측값이 존재하는 행을 삭제하기로 했다.

In [48]:
#AMOUNT_TYPE이 없는 행 제거
df = df.dropna(subset=["AMOUNT_TYPE"])

AMOUNT_NUM의 유효성: AMOUNT_TYPE이 삭제되거나 정리되면, 우리가 만든 AMOUNT_NUM을 안심하고 groupby 연산에 사용할 수 있습니다.

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94903 entries, 0 to 94922
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   RPT_DATE      94903 non-null  object 
 1   RPT_YR        94903 non-null  object 
 2   GEOGRAPHY     94903 non-null  object 
 3   SVC_DESC      94903 non-null  object 
 4   AMOUNT        94903 non-null  float64
 5   AMOUNT_ANNOT  94903 non-null  float64
 6   AMOUNT_TYPE   94903 non-null  object 
 7   AMOUNT_NUM    94903 non-null  float64
dtypes: float64(3), object(5)
memory usage: 6.5+ MB


In [53]:
# 카운티별 총 금액 상위 10개 출력
county_summary = df.groupby('GEOGRAPHY')['AMOUNT'].sum().sort_values(ascending=False)
print(county_summary.head(10))

GEOGRAPHY
STATE            54717357173.784
Los Angeles      23106285649.290
San Bernardino    4525430570.350
Riverside         3285196902.150
Santa Clara       3100309671.500
Orange            2459924849.340
San Diego         1909095907.090
Contra Costa      1753764794.100
Sacramento        1357521571.980
Kern              1163927416.890
Name: AMOUNT, dtype: float64


# 3. (선택) 합계 분석이 중요하다면, 숨겨진 값(ANNOT이 1 or 2인 경우)만 5로 보정
# 값이 달라지는 것이 걱정된다면 그냥 0으로 두셔도 되지만, 
# '데이터가 있음'을 표현하고 싶다면 5를 권장합니다.
import numpy as np
origin['AMOUNT_FINAL'] = np.where(
    (origin['AMOUNT_NUM'].isna()) & (origin['AMOUNT_ANNOT'] > 0), 
    5, # 혹은 사용자님의 의도에 따라 0
    origin['AMOUNT_NUM'].fillna(0)
)

### 